# AI-Enabled Detection of Exoplanets from Noisy Astronomical Light Curves

This notebook demonstrates the complete scientific pipeline for downloading, preprocessing, analyzing, and classifying stellar light curves to detect exoplanet transits. We use data from the **Transiting Exoplanet Survey Satellite (TESS)**.

### Pipeline Steps:
1. **Data Acquisition**: Download light curves from the MAST archive using `lightkurve`.
2. **Data Preprocessing**: Remove telemetry gaps (NaNs), filter outliers with sigma-clipping, and detrend stellar pulsations using a Savitzky-Golay filter.
3. **Transit Signal Detection**: Perform a Box Least Squares (BLS) search to locate periodic transit events and estimate orbital parameters.
4. **Feature Engineering**: Compute tabular features (skewness, kurtosis, noise estimates, odd-even differences, secondary eclipse check).
5. **Ensemble Machine Learning Inference**: Run predictions using our Random Forest, XGBoost, LightGBM, and PyTorch deep neural networks (1D CNN and CNN-LSTM).

In [ ]:
# Install dependencies if needed
# !pip install -r ../requirements.txt

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys

# Add parent directory to path to import src modules
sys.path.append(os.path.abspath('..'))

from src.data_acquisition import download_target_fits, PRESETS
from src.preprocessing import load_tess_fits, preprocess_light_curve
from src.transit_detection import perform_bls_search, get_phase_folded_lc, bin_folded_light_curve
from src.feature_engineering import extract_features
from src.models import ExoplanetClassifierPipeline, CLASSES
from src.confidence import run_monte_carlo_uncertainty, estimate_confidence_score
from src.visualization import *

print("All imports successful!")

## Step 1: Data Acquisition
We will download a real TESS target star: **L 98-59** (TIC 307210830, Sector 1), which is a known multi-planet system.

In [ ]:
tic_id = 307210830
sector = 1
fits_path = os.path.join("..", "data", "raw", f"tic_{tic_id}_sector_{sector}.fits")

if not os.path.exists(fits_path):
    print("Downloading TESS FITS file from MAST archive...")
    # Download and save in raw folder
    fits_path = download_target_fits(tic_id, sector, output_dir=os.path.join("..", "data", "raw"))
else:
    print(f"Using cached file: {fits_path}")

## Step 2: Data Preprocessing
Load the FITS file, extract PDCSAP flux, remove NaNs, filter cosmic-ray outliers using 3-sigma clipping, and flatten stellar variability using a Savitzky-Golay filter.

In [ ]:
df_raw = load_tess_fits(fits_path)
raw_time = df_raw['time'].values
raw_flux = df_raw['pdcsap_flux'].values

# Run preprocessing
t_clean, f_clean, f_detrend = preprocess_light_curve(
    raw_time, raw_flux, window_length=101, polyorder=2, sigma=3.0
)

# Plot preprocessing steps
trend = f_clean / f_detrend
fig = plot_raw_vs_cleaned(raw_time, raw_flux, t_clean, f_clean, trend=trend)
plt.show()

## Step 3: Box Least Squares (BLS) Transit Detection
Search for periodic dips. This returns the orbital period, transit depth, transit duration, mid-transit time ($t_0$), and signal-to-noise ratio.

In [ ]:
bls_results, bls_obj, results_raw = perform_bls_search(t_clean, f_detrend)

print(f"Detected Period: {bls_results['period']:.5f} days")
print(f"Transit Depth: {bls_results['depth']*1000:.3f} ppt (parts per thousand)")
print(f"Transit Duration: {bls_results['duration']*24:.2f} hours")
print(f"Signal-to-Noise Ratio (SNR): {bls_results['snr']:.2f}")

In [ ]:
# Plot BLS periodogram
fig_per = plot_bls_periodogram(bls_results["period_grid"], bls_results["power_grid"], bls_results["period"])
plt.show()

In [ ]:
# Phase-fold and bin the light curve
folded_phase, folded_flux = get_phase_folded_lc(t_clean, f_detrend, bls_results["period"], bls_results["t0"])
binned_phase, binned_flux = bin_folded_light_curve(folded_phase, folded_flux, n_bins=200)

# Plot folded transit profile
fig_fold = plot_folded_transit(folded_phase, folded_flux, binned_phase, binned_flux)
plt.show()

## Step 4: Feature Engineering & Ensemble Classification
We extract statistical moments, odd-even differences, and local noise estimates, and run inference using our classifier ensemble.

In [ ]:
features, binned_profile = extract_features(t_clean, raw_flux, f_clean, f_detrend)

# Print a subset of engineered features
print("Engineered Tabular Features:")
for k in ['detrended_std', 'detrended_skew', 'local_noise', 'odd_even_diff', 'secondary_depth']:
    print(f"- {k:<20}: {features[k]:.6f}")

In [ ]:
# Initialize and load pre-trained models
pipeline = ExoplanetClassifierPipeline(model_dir=os.path.join("..", "models"))

if pipeline.load_models():
    # Run predictions
    probs = pipeline.predict_probabilities(features, binned_profile)
    
    # Display probabilities bar chart
    fig_prob = plot_classification_probabilities(probs)
    plt.show()
    
    # Ensemble Prediction
    ensemble_probs = probs["Ensemble"]
    predicted_class = CLASSES[np.argmax(ensemble_probs)]
    
    conf_details = estimate_confidence_score(
        probs, bls_results["snr"], bls_results["depth"], features["local_noise"]
    )
    
    print(f"Ensemble Classification Result: {predicted_class.upper()}")
    print(f"Detection Confidence: {conf_details['confidence_score']:.2f}%")
    print(f"Significance: {conf_details['significance']}")
else:
    print("Pre-trained models not found in models/ folder. Run 'python -m src.evaluation' to train them.")

## Step 5: Monte Carlo Uncertainties
We perturb the light curve with Gaussian white noise scaled to the local noise floor and re-fit the transit using BLS to evaluate parameter uncertainties.

In [ ]:
print("Running Monte Carlo simulation (15 trials for speed)...")
mc_results = run_monte_carlo_uncertainty(
    t_clean, f_detrend, bls_results, features["local_noise"], n_trials=15
)

print(f"Orbital Period : {bls_results['period']:.5f} +/- {mc_results['period_err']:.5f} days")
print(f"Transit Depth  : {bls_results['depth']*1000:.3f} +/- {mc_results['depth_err']*1000:.3f} ppt")
print(f"Transit Duration: {bls_results['duration']*24:.2f} +/- {mc_results['duration_err']*24:.2f} hours")
print(f"Mid-Transit BJD: {bls_results['t0']:.4f} +/- {mc_results['t0_err']:.4f}")